In [1]:
CONFIG=dict(project_root='/home/featurize/work/biological_prior_validation',
            timeout=60,attempts=4,verify_existing=False,
            reserve_bytes=512*1024*1024,
            filenames=None)

In [2]:
from pathlib import Path
import urllib.request, urllib.error, urllib.parse
import json, re, hashlib, gzip, time, shutil, sys, platform
from datetime import datetime, timezone

ACCESSION='GSM7056149'
SOFT_URL='https://www.ncbi.nlm.nih.gov/geo/query/acc.cgi?acc=GSM7056149&targ=self&form=text&view=full'

def now():return datetime.now(timezone.utc).isoformat()
def atomic_json(path,obj):
    path=Path(path);tmp=path.with_name(path.name+'.tmp')
    tmp.write_text(json.dumps(obj,ensure_ascii=False,indent=2));tmp.replace(path)

def file_hash(path):
    h=hashlib.sha256()
    with Path(path).open('rb') as f:
        for b in iter(lambda:f.read(1048576),b''):h.update(b)
    return h.hexdigest()

def geo_files(text):
    if not re.search(r'^\^SAMPLE = GSM7056149\s*$',text,re.M):
        raise ValueError('GEO response is not GSM7056149 SOFT; possible HTML/sign-in/captcha page')
    urls=re.findall(r'^!Sample_supplementary_file_\d+ = (.+)$',text,re.M)
    files=[]
    for u in urls:
        u=u.strip().replace('ftp://','https://',1);parsed=urllib.parse.urlsplit(u)
        if parsed.scheme!='https' or parsed.hostname!='ftp.ncbi.nlm.nih.gov':
            raise ValueError('Unexpected supplementary host')
        name=Path(parsed.path).name
        if not name.startswith(ACCESSION+'_') or not name.endswith('.gz') or name in {x['filename'] for x in files}:
            raise ValueError('Unexpected/duplicate supplementary filename')
        files.append(dict(filename=name,url=u))
    if not files:raise ValueError('No supplementary files found')
    return files

def request(url,headers=None,method=None,timeout=60):
    hdr={'User-Agent':'GSM7056149-reproducible-data-audit/1.0','Accept-Encoding':'identity'}
    hdr.update(headers or {})
    return urllib.request.urlopen(urllib.request.Request(url,headers=hdr,method=method),timeout=timeout)

def probe(url,timeout=60):
    with request(url,method='HEAD',timeout=timeout) as r:
        size=r.headers.get('Content-Length')
        return dict(size_bytes=int(size) if size else None,etag=r.headers.get('ETag'),
                    last_modified=r.headers.get('Last-Modified'),accept_ranges=r.headers.get('Accept-Ranges'))

def gzip_prefix_check(path):
    with Path(path).open('rb') as f:magic=f.read(2)
    if magic!=b'\x1f\x8b':raise ValueError('Not gzip: possible downloaded HTML/error page')
    with gzip.open(path,'rb') as f:prefix=f.read(256)
    if not prefix:raise ValueError('Empty uncompressed prefix')
    return dict(gzip_validation='prefix_only_not_full_CRC',uncompressed_prefix_hex=prefix[:24].hex())

def same_version(old,new):
    # An absent validator provides no remote identity guarantee.
    for key in ['etag','last_modified','size_bytes']:
        if old.get(key) is not None and new.get(key) is not None and old[key]!=new[key]:return False
    return True

def download_one(item,raw_dir,previous=None,timeout=60,attempts=4,verify_existing=False,progress_every=10):
    raw_dir=Path(raw_dir);raw_dir.mkdir(parents=True,exist_ok=True)
    name=item['filename']
    if Path(name).name!=name:raise ValueError('Unsafe filename')
    dest=raw_dir/name;part=raw_dir/(name+'.part');side=raw_dir/(name+'.part.json')
    remote=item['remote'];total=remote.get('size_bytes')
    if dest.exists():
        if not previous or previous.get('status')!='complete':
            raise FileExistsError('Existing file lacks a completed manifest entry: '+str(dest))
        if previous.get('url')!=item['url'] or not same_version(previous.get('remote',{}),remote):
            raise ValueError('Existing file or GEO remote version changed; preserve it and inspect before replacing')
        if dest.stat().st_size!=previous['size_bytes'] or (total is not None and dest.stat().st_size!=total):
            raise ValueError('Existing file size mismatch')
        if verify_existing and file_hash(dest)!=previous['sha256']:raise ValueError('Existing SHA256 mismatch')
        return dict(previous,existing_check='size_and_remote_validators'+('_and_SHA256' if verify_existing else '_without_rehash'))
    if part.exists():
        if not side.exists():raise ValueError('Partial file has no version sidecar; preserve and inspect: '+str(part))
        saved=json.loads(side.read_text())
        if saved.get('url')!=item['url'] or not same_version(saved['remote'],remote):
            raise ValueError('Partial file remote version changed; do not append incompatible versions')
    atomic_json(side,dict(url=item['url'],remote=remote))
    last_error=None
    for attempt in range(attempts):
        offset=part.stat().st_size if part.exists() else 0
        if total is not None and offset>total:raise ValueError('Partial file exceeds remote size')
        try:
            if not (total is not None and offset==total):
                headers={}
                if offset:
                    headers['Range']=f'bytes={offset}-'
                    etag=remote.get('etag')
                    validator=etag if etag and not etag.startswith('W/') else remote.get('last_modified')
                    if validator:headers['If-Range']=validator
                with request(item['url'],headers,timeout=timeout) as response:
                    status=response.status
                    if status==206:
                        value=response.headers.get('Content-Range','')
                        match=re.fullmatch(r'bytes (\d+)-(\d+)/(\d+)',value)
                        if not match or int(match[1])!=offset:raise ValueError('Invalid resume Content-Range')
                        end=int(match[2]);response_total=int(match[3])
                        if end<offset or response_total<=end:raise ValueError('Invalid range bounds')
                        if total is not None and response_total!=total:raise ValueError('Remote total changed')
                        total=response_total
                        mode='ab' if offset else 'wb'
                    elif status==200:
                        # Server may ignore Range. Restart, never append the full body.
                        offset=0;mode='wb'
                        length=response.headers.get('Content-Length')
                        if length:
                            if total is not None and int(length)!=total:raise ValueError('Remote size changed')
                            total=int(length)
                    else:raise ValueError('Unexpected HTTP status: '+str(status))
                    h=hashlib.sha256()
                    if offset:
                        with part.open('rb') as f:
                            for b in iter(lambda:f.read(1048576),b''):h.update(b)
                    done=offset;tick=time.monotonic()
                    with part.open(mode) as f:
                        while True:
                            b=response.read(1048576)
                            if not b:break
                            f.write(b);h.update(b);done+=len(b)
                            if time.monotonic()-tick>=progress_every:
                                print(name, f'{done/1048576:.1f} MiB', f'/ {total/1048576:.1f} MiB' if total else '(total unknown)',flush=True)
                                tick=time.monotonic()
                    if total is not None and done!=total:raise IOError(f'Incomplete response: {done}/{total} bytes')
                    sha=h.hexdigest()
            else:sha=file_hash(part)
            validation=gzip_prefix_check(part)
            # Persist commit receipt BEFORE rename: can recover if interrupted between steps.
            receipt=dict(filename=name,url=item['url'],remote=remote,status='complete',
                         size_bytes=part.stat().st_size,sha256=sha,completed_utc=now(),
                         checksum_source='locally_computed_not_publisher_reference',**validation)
            atomic_json(raw_dir/(name+'.receipt.json'),receipt)
            part.replace(dest);side.unlink(missing_ok=True)
            print('COMPLETE',name,flush=True)
            return receipt
        except (urllib.error.URLError,TimeoutError,ConnectionError,IOError) as e:
            last_error=e
            if attempt+1<attempts:
                print('Retry',name,str(e),flush=True);time.sleep(min(2**attempt,5))
    raise RuntimeError('Download incomplete; partial retained: '+str(last_error))

def prepare(cfg):
    root=Path(cfg['project_root']);raw=root/'data/raw';audit=root/'data/metadata_audit'
    for sub in ['data/raw','data/metadata_audit','data/splits','data/processed','src','experiments','configs','results/raw','results/summary','figures','reports']:
        (root/sub).mkdir(parents=True,exist_ok=True)
    with request(SOFT_URL,timeout=cfg['timeout']) as r:text=r.read().decode('utf-8')
    (audit/'GSM7056149_geo.soft.txt').write_text(text)
    all_files=geo_files(text)
    selection=cfg.get('filenames')
    if selection is not None:
        unknown=set(selection)-{x['filename'] for x in all_files}
        if unknown:raise ValueError('Selected filenames absent from GEO: '+str(unknown))
        all_files=[x for x in all_files if x['filename'] in selection]
    for item in all_files:
        try:item['remote']=probe(item['url'],cfg['timeout'])
        except Exception as e:item['remote']=dict(size_bytes=None,etag=None,last_modified=None,probe_error=str(e))
    (audit/'download_plan.json').write_text(json.dumps(all_files,indent=2))
    (root/'configs/download_config.json').write_text(json.dumps(cfg,indent=2))
    (root/'results/raw/download_environment.json').write_text(json.dumps(dict(python=sys.version,platform=platform.platform(),accession=ACCESSION,utc=now(),geo_soft_sha256=hashlib.sha256(text.encode()).hexdigest()),indent=2))
    total=sum(x['remote']['size_bytes'] or 0 for x in all_files)
    free=shutil.disk_usage(root).free
    print('Destination:',raw)
    print('Known compressed total:',f'{total/1073741824:.2f} GiB; free {free/1073741824:.2f} GiB')
    print('Unknown-size files:',sum(x['remote']['size_bytes'] is None for x in all_files))
    for item in all_files:print(item['filename'],item['remote']['size_bytes'])
    # Existing complete/partial files are already included in used space, so reserve only remaining bytes.
    remaining=0
    for x in all_files:
        p=raw/x['filename'];part=raw/(x['filename']+'.part')
        used=p.stat().st_size if p.exists() else part.stat().st_size if part.exists() else 0
        remaining+=max(0,(x['remote']['size_bytes'] or 0)-used)
    if free<remaining+cfg['reserve_bytes']:raise OSError('Insufficient disk for known download sizes plus reserve')
    return all_files

def download_all(cfg,plan):
    root=Path(cfg['project_root']);raw=root/'data/raw';audit=root/'data/metadata_audit'
    manifest=audit/'download_manifest.json'
    state=json.loads(manifest.read_text()) if manifest.exists() else dict(accession=ACCESSION,files={})
    if state.get('accession')!=ACCESSION:raise ValueError('Manifest accession mismatch')
    for item in plan:
        name=item['filename'];previous=state['files'].get(name)
        # Recover a successful rename if a notebook was interrupted before manifest update.
        receipt=raw/(name+'.receipt.json')
        if (raw/name).exists() and receipt.exists() and (not previous or previous.get('status')!='complete'):
            previous=json.loads(receipt.read_text())
        try:
            result=download_one(item,raw,previous,cfg['timeout'],cfg['attempts'],cfg['verify_existing'])
            state['files'][name]=result
        except Exception as e:
            state['files'][name]=dict(filename=name,url=item['url'],status='failed',error=str(e),utc=now(),remote=item['remote'])
            print('FAILED',name,str(e),flush=True)
        state['updated_utc']=now();atomic_json(manifest,state)
    # Tables contain paths/checksums, never expression counts.
    import pandas as pd
    table=pd.DataFrame([dict(filename=name,status=v['status'],size_bytes=v.get('size_bytes'),sha256=v.get('sha256'),error=v.get('error')) for name,v in state['files'].items()])
    table.to_csv(audit/'download_manifest.csv',index=False)
    failed=[x['filename'] for x in plan if state['files'][x['filename']]['status']!='complete']
    if failed:raise RuntimeError('Some downloads failed; manifest and partial files saved. Re-run this cell after fixing the error: '+str(failed))
    return table


In [3]:
download_plan=prepare(CONFIG)


Destination: /home/featurize/work/biological_prior_validation/data/raw
Known compressed total: 10.39 GiB; free 74.59 GiB
Unknown-size files: 0
GSM7056149_sciPlexGxE_2_UMI.count.matrix.gz 6035886428
GSM7056149_sciPlexGxE_2_cell.annotations.txt.gz 115324478
GSM7056149_sciPlexGxE_2_gRNASampleSheet.txt.gz 28449
GSM7056149_sciPlexGxE_2_gRNATable_reads.out.txt.gz 656028109
GSM7056149_sciPlexGxE_2_gRNA_sequences.txt.gz 28786
GSM7056149_sciPlexGxE_2_gene.annotations.txt.gz 430161
GSM7056149_sciPlexGxE_2_hashTable.out.txt.gz 724394500
GSM7056149_sciPlexGxE_2_hash_sample_sheet.txt.gz 9213
GSM7056149_sciPlexGxE_2_hash_whitelist_rep1.txt.gz 3145
GSM7056149_sciPlexGxE_2_hash_whitelist_rep2.txt.gz 2979
GSM7056149_sciPlexGxE_2_preprocessed_cds.list.RDS.gz 3619987565


In [4]:
download_table=download_all(CONFIG,download_plan)
display(download_table)


,filename,status,size_bytes,sha256,error
0,GSM7056149_sciPlexGxE_2_UMI.count.matrix.gz,complete,6035886428,a1443d71dc890056bfb804d57bdef19d21942563ff088d...,None
1,GSM7056149_sciPlexGxE_2_cell.annotations.txt.gz,complete,115324478,7fffab7b4c323bdda5c21d6a71d499d9fde2748fc98e61...,None
2,GSM7056149_sciPlexGxE_2_gRNASampleSheet.txt.gz,complete,28449,39c8aaf26d4327219db844260d178d25b9b4ed0fea7449...,None
3,GSM7056149_sciPlexGxE_2_gRNATable_reads.out.tx...,complete,656028109,7fa62a7dd7225b931be064b32557d873a4639759ee5eb2...,None
4,GSM7056149_sciPlexGxE_2_gRNA_sequences.txt.gz,complete,28786,bb285a38bd9669cb12e2637d886bfde5fa566897ff10f8...,None
5,GSM7056149_sciPlexGxE_2_gene.annotations.txt.gz,complete,430161,60b62120c4d45fd49f4ef2fe778f969ce23df062c9dbe6...,None
6,GSM7056149_sciPlexGxE_2_hashTable.out.txt.gz,complete,724394500,04f22eb32e251478e08ecda657e0f68237fba53ce08913...,None
7,GSM7056149_sciPlexGxE_2_hash_sample_sheet.txt.gz,complete,9213,9282147247fee2445952099760097ff753a8186a855649...,None
8,GSM7056149_sciPlexGxE_2_hash_whitelist_rep1.tx...,complete,3145,8442ff836b9f0cd305338a8f6ee237042e12443059a8ee...,None
9,GSM7056149_sciPlexGxE_2_hash_whitelist_rep2.tx...,complete,2979,b28365aff9ef675325fe94bfb4bea89978b958b412db49...,None
